# Student 1 — Learning Style Score Integration + LSTM (All Four FSLSM Dimensions)

**Task:** Compute each learner's full FSLSM profile — **Input (Visual/Verbal)**,
**Processing (Active/Reflective)**, **Perception (Sensing/Intuitive)**, and
**Understanding (Sequential/Global)** — using the same binary-scoring methodology as
`Binary_Visual_Verbal_Scoring.pdf`, extended to the other three dimensions per the
`KDD Cup 2015 -> FSLSM Mapping` table. Integrate all four as *static* features alongside the
*sequential* clickstream data, train an LSTM (BiLSTM) dropout-prediction model on the fused
representation, and compare it against the sequential-only baseline.

Formatted as a **Google Colab notebook** (Drive mount + Kaggle dataset download), matching
the setup used across the team's other Colab notebooks.

**Design choices (unchanged, for consistency across the team):**

1. Same 2-week window, same train/test split as the rest of the pipeline.
2. All FSLSM scores are computed on the identical first-2-week window the sequence model
   reads — same-slice static + sequential features, no leakage from later behavior.
3. Fusion is a simple concatenation of a static-feature MLP embedding with the BiLSTM's final
   hidden state (Student 3's fusion-comparison notebook explores richer alternatives).
4. Baseline and FSLSM-augmented models are trained in the same run, same hyperparameters
   (3 epochs, Adam lr=1e-3, batch size 128, length-bucketed batching, BCE loss).

**Caveat, carried over from `FSLSM_All_Dimensions_Scoring.ipynb`:** only Input (Visual/Verbal)
is an exact, spec-verified formula from the PDF. Processing, Perception, and Understanding
required operationalizing qualifiers the mapping table doesn't define numerically ("high
dwell", "high retry", "broad", "low variance", "jumps") — see the assumptions table at the
end of this notebook before treating them as equally authoritative.

In [ ]:
import os

# 1. CONNECT TO GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

# 2. TELL COLAB WHERE YOUR KEY IS
# This points to the folder you just created in Phase 1
os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/My Drive/Kaggle"

# 3. DOWNLOAD KDD CUP 2015 DATA (AS A DATASET)
print("Downloading Data...")
# Use the -d flag for datasets instead of -c for competitions
!kaggle datasets download -d sst2023/kdd-cup-2015

# 4. UNZIP THE FILES
print("Unzipping...")
!unzip -q kdd-cup-2015.zip

# 5. LIST FILES (To confirm it worked)
print("Done! Here are your files:")
!ls

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import os

print("Loading raw data...")
log_train = pd.read_csv("train/train/log_train.csv")
log_test = pd.read_csv("test/test/log_test.csv")
object_df = pd.read_csv("object.csv")
truth_train = pd.read_csv("train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv("test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ==========================================
# FILTER FIRST 2 WEEKS  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

print(f"log_train (first 2 weeks): {log_train.shape}, log_test: {log_test.shape}")

In [ ]:
# ==========================================
# MERGE CATEGORY  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")

## FSLSM — all four dimensions

Each dimension follows the PDF's pattern: `Score = (A-B)/(A+B)`, tie -> second pole. See the
assumptions table at the end of this notebook for exactly how the qualifier-based poles
(dwell, retry, breadth, jumps) were operationalized.

In [ ]:
# Raw (unlogged) time gap between a learner's consecutive events, in seconds — needed for
# the dwell-time (Processing) and navigation-jump (Understanding) computations below.
def add_raw_time_gap(df):
    df = df.sort_values(['enrollment_id', 'time']).copy()
    df['time_gap_sec'] = df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0)
    return df

log_train = add_raw_time_gap(log_train)
log_test = add_raw_time_gap(log_test)


def compute_input_dimension(df):
    # V_i = video_i + access_i ; R_i = wiki_i + discussion_i  (verbatim from the PDF)
    counts = df.groupby(['enrollment_id', 'event']).size().unstack(fill_value=0)
    for col in ['video', 'access', 'wiki', 'discussion']:
        if col not in counts.columns:
            counts[col] = 0
    V = counts['video'] + counts['access']
    R = counts['wiki'] + counts['discussion']
    total = V + R
    S = np.where(total > 0, (V - R) / total, 0.0)
    label = np.where(V > R, 1, 0)  # 1 = Visual, 0 = Verbal (ties -> Verbal)
    return pd.DataFrame({'student_id': counts.index, 'input_S': S, 'input_label': label}).reset_index(drop=True)


def compute_processing_dimension(df):
    # Active_i = count(problem) + count(discussion) — literal count.
    active_counts = df[df['event'].isin(['problem', 'discussion'])].groupby('enrollment_id').size()
    # Reflective_i = summed dwell time (time_gap_sec) on video/access events — operationalizes "high dwell".
    reflective_dwell = df[df['event'].isin(['video', 'access'])].groupby('enrollment_id')['time_gap_sec'].sum()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['A'] = active_counts
    out['Rf'] = reflective_dwell
    out = out.fillna(0.0)
    total = out['A'] + out['Rf']
    S = np.where(total > 0, (out['A'] - out['Rf']) / total, 0.0)
    label = np.where(out['A'] > out['Rf'], 1, 0)  # 1 = Active, 0 = Reflective (ties -> Reflective)
    out = out.reset_index()
    out['processing_S'] = S
    out['processing_label'] = label
    return out[['student_id', 'processing_S', 'processing_label']]


def compute_perception_dimension(df):
    # Sensing_i: repeat visits to the SAME object during 'problem' events, beyond the first (= retries).
    problem_df = df[df['event'] == 'problem']
    visits_per_object = problem_df.groupby(['enrollment_id', 'object']).size()
    retries_per_object = (visits_per_object - 1).clip(lower=0)
    sensing_retries = retries_per_object.groupby('enrollment_id').sum()
    # Intuitive_i: count(wiki) + breadth of navigate exploration (# distinct objects reached via navigate).
    wiki_counts = df[df['event'] == 'wiki'].groupby('enrollment_id').size()
    navigate_breadth = df[df['event'] == 'navigate'].groupby('enrollment_id')['object'].nunique()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['Sn'] = sensing_retries
    out['wiki'] = wiki_counts
    out['breadth'] = navigate_breadth
    out = out.fillna(0.0)
    out['In'] = out['wiki'] + out['breadth']
    total = out['Sn'] + out['In']
    S = np.where(total > 0, (out['Sn'] - out['In']) / total, 0.0)
    label = np.where(out['Sn'] > out['In'], 1, 0)  # 1 = Sensing, 0 = Intuitive (ties -> Intuitive)
    out = out.reset_index()
    out['perception_S'] = S
    out['perception_label'] = label
    return out[['student_id', 'perception_S', 'perception_label']]


def _count_jumps(gaps):
    gaps = np.asarray(gaps)
    if len(gaps) < 4:
        return 0
    q1, q3 = np.percentile(gaps, [25, 75])
    threshold = q3 + 1.5 * (q3 - q1)
    return int((gaps > threshold).sum())


def compute_understanding_dimension(df):
    nav_df = df[df['event'] == 'navigate'].sort_values(['enrollment_id', 'time']).copy()
    nav_df['nav_gap'] = nav_df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0)
    nav_counts = nav_df.groupby('enrollment_id').size()
    jump_counts = nav_df.groupby('enrollment_id')['nav_gap'].apply(_count_jumps)
    page_close_counts = df[df['event'] == 'page_close'].groupby('enrollment_id').size()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['nav_total'] = nav_counts
    out['jumps'] = jump_counts
    out['page_close'] = page_close_counts
    out = out.fillna(0.0)
    out['Sq'] = (out['nav_total'] - out['jumps']).clip(lower=0)
    out['Gl'] = out['jumps'] + out['page_close']
    total = out['Sq'] + out['Gl']
    S = np.where(total > 0, (out['Sq'] - out['Gl']) / total, 0.0)
    label = np.where(out['Sq'] > out['Gl'], 1, 0)  # 1 = Sequential, 0 = Global (ties -> Global)
    out = out.reset_index()
    out['understanding_S'] = S
    out['understanding_label'] = label
    return out[['student_id', 'understanding_S', 'understanding_label']]


def compute_all_fslsm_dimensions(df):
    out = compute_input_dimension(df) \
        .merge(compute_processing_dimension(df), on='student_id', how='outer') \
        .merge(compute_perception_dimension(df), on='student_id', how='outer') \
        .merge(compute_understanding_dimension(df), on='student_id', how='outer')
    return out.fillna(0.0)

fslsm_train = compute_all_fslsm_dimensions(log_train)
fslsm_test = compute_all_fslsm_dimensions(log_test)

print("FSLSM feature summary (train):")
print(fslsm_train.describe())
for dim in ['input', 'processing', 'perception', 'understanding']:
    print(f"  {dim}_label distribution: {fslsm_train[f'{dim}_label'].value_counts().to_dict()}")

os.makedirs("data/processed", exist_ok=True)
fslsm_train.to_csv("data/processed/fslsm_full_train.csv", index=False)
fslsm_test.to_csv("data/processed/fslsm_full_test.csv", index=False)
print("\nSaved data/processed/fslsm_full_train.csv / fslsm_full_test.csv (for Student 2 & 3 to reuse).")

## Sequential feature engineering & sequence building

Identical to `02_Data_Preprocessing.ipynb`.

In [ ]:
# ==========================================
# FEATURE ENGINEERING  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)

print(f"train_df: {train_df.shape}, test_df: {test_df.shape}")

In [ ]:
# ==========================================
# MERGE ALL FOUR STATIC FSLSM DIMENSIONS ONTO THE SEQUENCE TABLES
# ==========================================
FSLSM_COLS = ['input_S', 'input_label', 'processing_S', 'processing_label',
              'perception_S', 'perception_label', 'understanding_S', 'understanding_label']

train_df = train_df.merge(fslsm_train, on="student_id", how="left")
test_df = test_df.merge(fslsm_test, on="student_id", how="left")

missing_train = train_df[FSLSM_COLS].isna().any(axis=1).sum()
missing_test = test_df[FSLSM_COLS].isna().any(axis=1).sum()
print(f"Rows missing FSLSM features -> train: {missing_train}, test: {missing_test}")
# Every student here comes from the same filtered logs used to build fslsm_train/test,
# so this should be 0; fillna(0) is a safety net.
train_df[FSLSM_COLS] = train_df[FSLSM_COLS].fillna(0)
test_df[FSLSM_COLS] = test_df[FSLSM_COLS].fillna(0)

In [ ]:
# ==========================================
# ENCODING (source / event / category streams) — identical to 02_Data_Preprocessing.ipynb
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat
    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)

In [ ]:
# ==========================================
# STATIC FEATURE VECTOR: 4 continuous S-scores (already in [-1, 1], no scaling needed)
# + 4 binary labels  ->  8-dim static vector per learner
# ==========================================
train_static = train_df[FSLSM_COLS].values.astype(np.float32)
test_static = test_df[FSLSM_COLS].values.astype(np.float32)

STATIC_DIM = train_static.shape[1]
print(f"Static feature vector per learner ({STATIC_DIM}-dim): {FSLSM_COLS}")

In [ ]:
# ==========================================
# VARIABLE-LENGTH SEQUENCES + aligned static feature vectors
# ==========================================
def build_variable_length(df, static_arr):
    src_list = [torch.LongTensor(s) for s in df["src"]]
    evt_list = [torch.LongTensor(e) for e in df["evt"]]
    cat_list = [torch.LongTensor(c) for c in df["cat"]]
    gap_list = [torch.FloatTensor(g) for g in df["gaps"]]
    static_list = [torch.FloatTensor(static_arr[i]) for i in range(len(df))]
    y = torch.FloatTensor(df["dropout"].values)
    ids = df["student_id"].values
    return src_list, evt_list, cat_list, gap_list, static_list, y, ids

print("Building variable-length sequences (sequential streams + 8-dim FSLSM static vector)...")
src_train, evt_train, cat_train, gap_train, static_train_t, y_train, train_ids = build_variable_length(train_df, train_static)
src_test, evt_test, cat_test, gap_test, static_test_t, y_test, test_ids = build_variable_length(test_df, test_static)

lengths_train = [len(s) for s in src_train]
print(f"Train sequence lengths -> min: {min(lengths_train)}, max: {max(lengths_train)}, mean: {np.mean(lengths_train):.1f}")

## Dataset / collation / batching

The static vector is 8-dim (all four FSLSM dimensions).

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, static, y=None):
        self.src, self.evt, self.cat, self.gaps, self.static = src, evt, cat, gaps, static
        self.y = y

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        if self.y is None:
            return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.static[idx]
        return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.static[idx], self.y[idx]


def collate_fn(batch):
    has_labels = len(batch[0]) == 6
    if has_labels:
        src, evt, cat, gaps, static, y = zip(*batch)
    else:
        src, evt, cat, gaps, static = zip(*batch)

    lengths = torch.LongTensor([len(s) for s in src])

    src = pad_sequence(src, batch_first=True, padding_value=0)
    evt = pad_sequence(evt, batch_first=True, padding_value=0)
    cat = pad_sequence(cat, batch_first=True, padding_value=0)
    gaps = pad_sequence(gaps, batch_first=True, padding_value=0.0)
    static = torch.stack(static)  # fixed-length, no padding needed

    if has_labels:
        y = torch.stack(y)
        return src, evt, cat, gaps, static, lengths, y
    return src, evt, cat, gaps, static, lengths


class BucketBatchSampler(Sampler):
    # Identical to 03_Model_Training_Evaluation.ipynb: groups similar-length
    # sequences into the same batch, then shuffles batch ORDER (not batch contents).
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.argsort(self.lengths)
        batches = [indices[i:i+self.batch_size].tolist() for i in range(0, len(indices), self.batch_size)]
        if self.shuffle:
            np.random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size


BATCH_SIZE = 128
train_lengths = [len(s) for s in src_train]
test_lengths = [len(s) for s in src_test]

train_sampler = BucketBatchSampler(train_lengths, batch_size=BATCH_SIZE, shuffle=True)
test_sampler = BucketBatchSampler(test_lengths, batch_size=BATCH_SIZE, shuffle=False)

train_loader = DataLoader(SeqDataset(src_train, evt_train, cat_train, gap_train, static_train_t, y_train),
                           batch_sampler=train_sampler, collate_fn=collate_fn)
test_loader = DataLoader(SeqDataset(src_test, evt_test, cat_test, gap_test, static_test_t, y_test),
                          batch_sampler=test_sampler, collate_fn=collate_fn)

## Models

- **`BiLSTMModel`** — sequential-only baseline (unchanged from `03`).
- **`BiLSTM_FSLSM_Model`** — same BiLSTM backbone, plus a small MLP over the 8-dim
  all-dimension FSLSM vector, fused by concatenation with the BiLSTM's final hidden state.

In [ ]:
class BiLSTMModel(nn.Module):
    # Sequential-only baseline (identical to 03_Model_Training_Evaluation.ipynb).
    def __init__(self):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)
        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, src, evt, cat, gaps, lengths, static=None):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)
        return torch.sigmoid(self.fc(h_cat)).squeeze()


class BiLSTM_FSLSM_Model(nn.Module):
    # Sequential BiLSTM backbone (same as baseline) fused with the 8-dim all-dimension
    # static FSLSM branch via concatenation before the classifier head.
    def __init__(self, static_dim):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)
        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)

        self.static_mlp = nn.Sequential(
            nn.Linear(static_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )
        self.fc = nn.Linear(128 + 16, 1)

    def forward(self, src, evt, cat, gaps, lengths, static):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)

        static_emb = self.static_mlp(static)           # (batch, 16)
        fused = torch.cat([h_cat, static_emb], dim=1)  # (batch, 144)
        return torch.sigmoid(self.fc(fused)).squeeze()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

## Shared training / evaluation loop

Identical hyperparameters for both models (3 epochs, Adam lr=1e-3, BCE loss, same
length-bucketed batches) so the comparison isolates the effect of the FSLSM static features.

In [ ]:
def train_and_evaluate(model, model_name, uses_static, epochs=3, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, evt, cat, gaps, static, lengths, labels in train_loader:
            src, evt, cat, gaps, static, labels = [t.to(device) for t in (src, evt, cat, gaps, static, labels)]
            optimizer.zero_grad()
            outputs = model(src, evt, cat, gaps, lengths, static) if uses_static else model(src, evt, cat, gaps, lengths)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[{model_name}] Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    preds = []
    with torch.no_grad():
        for src, evt, cat, gaps, static, lengths, _ in test_loader:
            src, evt, cat, gaps, static = [t.to(device) for t in (src, evt, cat, gaps, static)]
            outputs = model(src, evt, cat, gaps, lengths, static) if uses_static else model(src, evt, cat, gaps, lengths)
            preds.extend(outputs.cpu().numpy())

    ordered_ids = []
    for batch in BucketBatchSampler(test_lengths, batch_size=BATCH_SIZE, shuffle=False):
        ordered_ids.extend([test_ids[i] for i in batch])

    eval_df = pd.DataFrame({"student_id": ordered_ids, "pred": preds}).merge(truth_test, on="student_id")
    eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "Precision": precision_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "Recall": recall_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "F1": f1_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "AUC": roc_auc_score(eval_df["dropout"], eval_df["pred"]),
    }
    print(f"\n[{model_name}] Test metrics:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"  {k:<10}: {v:.4f}")
    return metrics, eval_df

## Train the baseline (sequential-only) model

In [ ]:
torch.manual_seed(SEED)
baseline_model = BiLSTMModel()
baseline_metrics, baseline_eval_df = train_and_evaluate(
    baseline_model, model_name="Baseline BiLSTM (sequential only)", uses_static=False
)

## Train the FSLSM-augmented model (all four dimensions)

In [ ]:
torch.manual_seed(SEED)
fslsm_model = BiLSTM_FSLSM_Model(static_dim=STATIC_DIM)
fslsm_metrics, fslsm_eval_df = train_and_evaluate(
    fslsm_model, model_name="BiLSTM + FSLSM all-dimension static features (concat fusion)", uses_static=True
)

## Comparison table & handoff to the team

Saved to `data/results/student1_lstm_fslsm_results.csv` for Student 3's full comparison.

In [ ]:
comparison_df = pd.DataFrame([baseline_metrics, fslsm_metrics])
comparison_df["Fusion Approach"] = ["N/A (no static features)", "Concatenation (intermediate fusion)"]
comparison_df["Input Features"] = [
    "Sequential (source/event/category tokens + time gaps)",
    "Sequential + static FSLSM all-dimension features (Input, Processing, Perception, "
    "Understanding: S-score + binary label each, 8-dim total)"
]
comparison_df = comparison_df[["Model", "Fusion Approach", "Input Features", "Accuracy", "Precision", "Recall", "F1", "AUC"]]

print(comparison_df.to_string(index=False))

improvement = fslsm_metrics["AUC"] - baseline_metrics["AUC"]
print(f"\nAUC change from adding all four FSLSM static dimensions: {improvement:+.4f}")

os.makedirs("data/results", exist_ok=True)
comparison_df.to_csv("data/results/student1_lstm_fslsm_results.csv", index=False)
print("Saved comparison table to data/results/student1_lstm_fslsm_results.csv")

## Notes for the team / assumptions summary

- `data/processed/fslsm_full_train.csv` / `fslsm_full_test.csv` now hold all four dimensions'
  `S` scores and binary labels per `student_id` for the first-2-week window. Student 2's
  Transformer and Student 3's fusion notebook should read these files (not the earlier
  Visual/Verbal-only `fslsm_train.csv`) so every experiment scores learning style identically.
- Only **Input** is an exact, spec-verified formula from the PDF. **Processing**,
  **Perception**, and **Understanding** required operationalizing qualifiers the mapping
  table doesn't define numerically:
  - Processing/Reflective "high dwell" → summed inter-event time on video/access
  - Perception/Sensing "high retry" → repeat visits to the same problem-object
  - Perception/Intuitive "broad" → wiki count + distinct navigate-object breadth
  - Understanding "low variance"/"jumps" → per-learner Tukey-outlier navigate time-gaps
  These are documented, defensible choices — not settled definitions — and should be
  flagged as such in any write-up.
- The fusion here (concatenate an 8-dim static MLP embedding with the BiLSTM's final hidden
  state) is one reasonable integration. Student 3's notebook should treat this row as
  "intermediate/concatenation fusion, all 4 FSLSM dimensions" and compare it against
  early-fusion, late-fusion, and attention-based variants on the same split/metrics.